<a href="https://colab.research.google.com/github/emt0147t/AAIC/blob/main/semi_supervised_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import kagglehub
path = kagglehub.dataset_download("vahidehdashti/movielens")

Using Colab cache for faster access to the 'movielens' dataset.


In [7]:
import pandas as pd
import os

data_path = os.path.join(path, "ml-100k", "u.data")
item_path = os.path.join(path, "ml-100k", "u.item")

data_cols = ['user_id', 'item_id', 'rating', 'timestamp']
df_ratings = pd.read_csv(data_path, sep='\t', names=data_cols)


item_cols = ['item_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL', 'unknown', 'Action', 'Adventure', 'Animation', 'Childrens', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
df_movies = pd.read_csv(item_path, sep='|', names=item_cols, encoding='latin-1')

print("5 dòng đầu của ma trận đánh giá")
display(df_ratings.head())

5 dòng đầu của ma trận đánh giá


,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [8]:

user_movie_matrix = df_ratings.pivot(index='user_id', columns='item_id', values='rating')

print(f"kích thước ma trận (số người dùng x số bộ phim): {user_movie_matrix.shape}")

total_cells = user_movie_matrix.shape[0] * user_movie_matrix.shape[1]
filled_cells = user_movie_matrix.count().sum()
sparsity = 100 - (filled_cells / total_cells * 100)
print(f"tỷ lệ ô trống (Sparsity): {sparsity:.2f}%\n")

display(user_movie_matrix.iloc[:5, :10])

kích thước ma trận (số người dùng x số bộ phim): (943, 1682)
tỷ lệ ô trống (Sparsity): 93.70%



item_id,1,2,3,4,5,6,7,8,9,10
user_id,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:

user_similarity_pearson = user_movie_matrix.T.corr(method='pearson')

print("ma trận tương đồng 5 người dùng đầu tiên")
display(user_similarity_pearson.iloc[:5, :5])


target_user = 1

top_neighbors = user_similarity_pearson[target_user].drop(target_user).sort_values(ascending=False).head(5)

print(f"top 5 user cùng tương đồng người dùng {target_user} ---")
print(top_neighbors)

ma trận tương đồng 5 người dùng đầu tiên


user_id,1,2,3,4,5
user_id,,,,,
1,1.000000,0.160841,0.11278,0.500000,0.420809
2,0.160841,1.000000,0.06742,0.148522,0.327327
3,0.112780,0.067420,1.00000,-0.262600,NaN
4,0.500000,0.148522,-0.26260,1.000000,1.000000
5,0.420809,0.327327,NaN,1.000000,1.000000


top 5 user cùng tương đồng người dùng 1 ---
user_id
273    1.0
351    1.0
811    1.0
511    1.0
866    1.0
Name: 1, dtype: float64


In [10]:
import numpy as np

target_user_ratings = user_movie_matrix.loc[target_user]
unseen_movies = target_user_ratings[target_user_ratings.isna()].index


recommendations = []


for movie_id in unseen_movies:
    total_score = 0
    total_weight = 0

    for neighbor_id, similarity in top_neighbors.items():

        neighbor_rating = user_movie_matrix.loc[neighbor_id, movie_id]


        if not np.isnan(neighbor_rating):
            total_score += similarity * neighbor_rating
            total_weight += similarity


    if total_weight > 0:
        predicted_rating = total_score / total_weight
        recommendations.append({'item_id': movie_id, 'predicted_rating': predicted_rating})

df_recommendations = pd.DataFrame(recommendations)
df_top3 = df_recommendations.sort_values(by='predicted_rating', ascending=False).head(3)

df_final = pd.merge(df_top3, df_movies[['item_id', 'movie_title']], on='item_id', how='left')

print(f"--- top 3 phim đề xuất cho người dùng {target_user} ---")
display(df_final[['movie_title', 'predicted_rating']])

--- top 3 phim đề xuất cho người dùng 1 ---


,movie_title,predicted_rating
0,Anna Karenina (1997),5.0
1,"Big Lebowski, The (1998)",5.0
2,"Postman, The (1997)",5.0
